# Data segmentation

This notebook will attempt to segment the documents stored ind dictionaris stored in Data\ extraction/clean-data/..

In [1]:
from congreso import congreso as c 
import pandas as pd
from matplotlib import pyplot as plt
import json
import os
import re

In [2]:
folder_path = os.path.join('..', 'Data extraction', 'clean-data')  # Goes one level up, then into 'Data extraction/clean-data'
documents = {}
years = [2003, 2013, 2023]
for year in years:
    file_path = os.path.join(folder_path, f"d_{year}.json")
    
    if os.path.exists(file_path):
        with open(file_path, "r", encoding="utf-8") as f:
            documents[year] = json.load(f)
        print(f"Loaded data for year {year} into documents[{year}]")
    else:
        print(f"File for year {year} not found at {file_path}")


Loaded data for year 2003 into documents[2003]
Loaded data for year 2013 into documents[2013]
Loaded data for year 2023 into documents[2023]


In [3]:
d_2003 = documents[2003]
d_2013 = documents[2013]
d_2023 = documents[2023]


Now let's start with the actual segmentation, first, some segmentation for a single doc to define the formats.

## Segmentation

### End start segmentation

In [4]:
def segment_docs(docs):
    segmented_docs = []
    successfully_segmented_count = 0

    # Define start patterns (handles cases where there's no space after a period)
    start_patterns = [
        r"(?:\.|\s|^)El señor (PRESIDENTE|VICEPRESIDENTE)(?: \([^\)]*\))?:",  
        r"(?:\.|\s|^)La señora (PRESIDENTA|VICEPRESIDENTA)(?: \([^\)]*\))?:", 
    ]
    
    for doc in docs:
        segmented_doc = {key: value for key, value in doc.items() if key != 'text'}
        text = doc.get('texto', '')

        # Find the first occurrence of any start pattern
        start_match = None
        start_positions = []

        for pattern in start_patterns:
            matches = list(re.finditer(pattern, text))
            if matches:
                start_positions.append(matches[0].start())  # Get the first occurrence of each pattern

        if start_positions:
            start_match = min(start_positions)  # Get the earliest match
        
        # Use the last period in the text as fallback end
        end_match = text.rfind('.')
        if end_match == -1:
            end_match = len(text)  # Fallback to full text if no period found
        
        # Extract the segment if a start is found
        if start_match is not None:
            segmented_text = text[start_match:end_match]
            segmented_doc['texto'] = segmented_text
            segmented_docs.append(segmented_doc)
            successfully_segmented_count += 1
        else:
            segmented_doc['texto'] = ''  # Leave empty if no valid start is found
            segmented_docs.append(segmented_doc)

    total_success = successfully_segmented_count / len(docs) if docs else 0
    return segmented_docs, total_success


In [5]:
ds_2003, success = segment_docs(d_2003)
print(success)
ds_2013, success = segment_docs(d_2013)
print(success)
ds_2023, success = segment_docs(d_2023)
print(success)


1.0
1.0
1.0


### Segment interventions

In [6]:
def normalize_role(raw_title: str, gender: str) -> tuple[str, str]:
    raw_title = raw_title.strip().upper()

    if "PRESIDENTE DEL GOBIERNO" in raw_title or "PRESIDENTA DEL GOBIERNO" in raw_title:
        return ("Presidente" if gender == "M" else "Presidenta", "Presidenta del Gobierno" if gender == "F" else "Presidente del Gobierno")

    if raw_title.startswith("VICEPRESIDENTE") or raw_title.startswith("VICEPRESIDENTA"):
        if raw_title.strip() in ["VICEPRESIDENTE", "VICEPRESIDENTA"]:
            return ("Vicepresidente" if gender == "M" else "Vicepresidenta", "Vicepresidenta de la Cámara" if gender == "F" else "Vicepresidente de la Cámara")
        else:
            return ("Vicepresidente" if gender == "M" else "Vicepresidenta", "Vicepresidenta del Gobierno" if gender == "F" else "Vicepresidente del Gobierno")

    if raw_title.startswith("MINISTRO") or raw_title.startswith("MINISTRA"):
        return ("Ministro" if gender == "M" else "Ministra", raw_title.title())

    if raw_title in ["PRESIDENTE", "PRESIDENTA"]:
        return ("Presidente" if gender == "M" else "Presidenta", "Presidenta de la Cámara" if gender == "F" else "Presidente de la Cámara")

    return (None, None)


In [7]:
def extract_interventions(doc):
    interventions = []
    text = doc.get("texto", "")
    date = doc.get("fecha", "unknown_date")
    doc_id = doc.get("pdf_url", "unknown_doc")

    intervention_pattern = r"(?:(?<=^)|(?<=[\.\n!?])|\s) ?((El|La)? ?(señor|señora) ([A-ZÁÉÍÓÚÑÜÀ-ÿ ,.\-]+?)(?: \(([^()]+)\))?):"

    if not text:
        print(f"Warning: 'texto' field is empty in {doc_id}")
        return interventions

    matches = list(re.finditer(intervention_pattern, text))
    if not matches:
        print(f"No interventions found in {doc_id}")
        return interventions

    for i, match in enumerate(matches):
        start = match.start(1)
        end = matches[i + 1].start(1) if i + 1 < len(matches) else len(text)
        intervention_text = text[start:end].strip()

        # Remove the "El señor XXX:" part
        intervention_text = intervention_text[len(match.group(1)) + 1:].strip()

        gender_tag = match.group(3)
        gender = "M" if gender_tag == "señor" else "F" if gender_tag == "señora" else "N"

        raw_title = match.group(4).strip()
        alt_author = match.group(5).strip() if match.group(5) else None

        if raw_title in ["PRESIDENTA", "PRESIDENTE"] and not alt_author:
            author = raw_title
            charge = "Presidenta" if gender == "F" else "Presidente"
            raw_charge = "Presidenta de la Cámara" if gender == "F" else "Presidente de la Cámara"
        else:
            author = alt_author if alt_author else raw_title
            if raw_title and author.upper() == raw_title.upper():
                charge, raw_charge = None, None
            else:
                charge, raw_charge = normalize_role(raw_title, gender)

        # Normalize author name
        author = " ".join([part.capitalize() for part in author.split()])

        interventions.append({
            "id": f"{date}.i{i + 1}",
            "autor": author,
            "raw_charge": raw_charge,
            "charge": charge,
            "gender": gender,
            "text": intervention_text,
            "date": date,
            "document_id": doc_id,
            "num_int": i + 1
        })

    return interventions


In [ ]:
pd.set_option("display.max_rows", None)  # Show all rows
ints = extract_interventions(ds_2023[0])
num_interventions = len(ints)

print(f"Number of interventions: {num_interventions}")

# Convert to DataFrame and exclude 'text' column
df = pd.DataFrame(ints).drop(columns=["text"])

# Display as a table
display(df)


In [9]:
def segment_all_years(docs_by_year):
    all_interventions = []

    for year, docs in docs_by_year.items():
        print(f"Processing {len(docs)} documents from {year}...")
        for doc in docs:
            interventions = extract_interventions(doc)
            all_interventions.extend(interventions)

    df = pd.DataFrame(all_interventions)
    df.to_csv("all_segmented_interventions.csv", index=False)
    print(f"Done. Total interventions: {len(df)}")
    return df


In [10]:
docs_by_year = {
    "2003": ds_2003,
    "2013": ds_2013,
    "2023": ds_2023
}

df_all = segment_all_years(docs_by_year)


Processing 84 documents from 2003...
Processing 79 documents from 2013...
Processing 46 documents from 2023...
Done. Total interventions: 36379
